# CastellanoSQL

## Objetivo

Crear un traductor que le permita a cualquier persona consultar una base de datos usando español sencillo, sin necesidad de saber programar en SQL. Por ejemplo, si el usuario escribe "buscar productos donde precio sea menor a 5000", el programa entiende la orden, comprueba que esté bien escrita y la convierte en una consulta técnica real, "SELECT * FROM productos WHERE precio < 5000.

## Alcance

* Reconocer y validar consultas simples de selección (`buscar <tabla>`).
* Aplicar condiciones de filtrado lógico (`donde <campo> sea igual a/mayor a/menor a <valor>`).
* Soportar conectores lógicos secuenciales (`y`, `o`).
* Permitir ordenamiento de resultados (`ordenar por <campo> ascendente/descendente`).
* Generar la sintaxis estructurada SQL correspondiente (`SELECT ... FROM ... WHERE ... ORDER BY ...`).

## Especificaciones léxicas

| Token | Lexema / Regla | Descripción |
| :--- | :--- | :--- |
| `BUSCAR` | `buscar` | Palabra clave para indicar la entidad o tabla a consultar (`SELECT`). |
| `DONDE` | `donde` | Palabra clave para iniciar las cláusulas de filtrado (`WHERE`). |
| `AGRUPAR` | `agrupar` | Palabra clave para iniciar la agrupación (`GROUP BY`). |
| `TENIENDO` | `teniendo` | Palabra clave para filtrar grupos (`HAVING`). |
| `ORDENAR` | `ordenar` | Palabra clave para iniciar el ordenamiento (`ORDER BY`). |
| `POR` | `por` | Conector para cláusulas de agrupación y ordenamiento. |
| `DE` | `de` | Preposición conectora para funciones de agregación. |
| `ASCENDENTE` | `ascendente` \| `asc` | Especificador de orden ascendente (`ASC`). |
| `DESCENDENTE` | `descendente` \| `desc` | Especificador de orden descendente (`DESC`). |
| `Y` | `y` | Conector lógico AND (`AND`). |
| `O` | `o` | Conector lógico OR (`OR`). |
| `FUNCION` | `contar` \| `promedio` \| `suma` \| `maximo` \| `minimo` | Funciones de agregación SQL (`COUNT( ), AVG( ), SUM( ), MAX( ), MIN( )`)|
| `OPERADOR` | `sea mayor a` \| `mayor a` \| `sea menor a` \| `menor a` \| `sea igual a` \| `igual a` \| `sea diferente de` \| `diferente de` | Frases compuestas para comparaciones relacionales (`>`, `<`, `=`, `!=`). |
| `IDENTIFICADOR` | `[a-zA-Z_][a-zA-Z0-9_]*` | Nombre de la entidad, tabla o columna. |
| `NUMERO` | `\d+(\.\d+)?` | Valores numéricos enteros o decimales. |
| `CADENA` | `"[^"]*"` | Textos o valores literales entre comillas dobles. |

## Especificaciones sintácticas

```text
<programa>          ::= <consulta>

<consulta>          ::= BUSCAR <seleccion> IDENTIFICADOR <clausula_donde> <clausula_agrupar> <clausula_ordenar>

<seleccion>         ::= FUNCION DE | ε

<clausula_donde>    ::= DONDE <condiciones> | ε

<clausula_agrupar>  ::= AGRUPAR POR IDENTIFICADOR <clausula_teniendo> | ε

<clausula_teniendo> ::= TENIENDO <condiciones_grupo> | ε

<clausula_ordenar>  ::= ORDENAR POR IDENTIFICADOR <direccion> | ε

<condiciones>       ::= <condiciones> <conector_logico> <condicion> | <condicion>
<condicion>         ::= IDENTIFICADOR OPERADOR <valor>

<condiciones_grupo> ::= <condiciones_grupo> <conector_logico> <condicion_grupo> | <condicion_grupo>
<condicion_grupo>   ::= FUNCION OPERADOR <valor> | IDENTIFICADOR OPERADOR <valor>

<conector_logico>   ::= Y | O
<valor>             ::= NUMERO | CADENA
<direccion>         ::= ASCENDENTE | DESCENDENTE | ε

## Especificaciones semánticas


1. **Identificación de la Entidad y Proyección (`<consulta>`, `<seleccion>`):** 
   * La cláusula `BUSCAR` seguida de un `IDENTIFICADOR` determina la tabla o colección fuente de la consulta (equivalente al `FROM` en SQL).
   * Si se omite `<seleccion>` ($\varepsilon$), la consulta proyecta todos los atributos (`*`). Si se especifica `FUNCION DE`, aplica la función de agregación correspondiente (`contar` $\rightarrow$ `COUNT(*)`, `promedio` $\rightarrow$ `AVG(*)`, `suma` $\rightarrow$ `SUM(*)`, `maximo` $\rightarrow$ `MAX(*)`, `minimo` $\rightarrow$ `MIN(*)`).

2. **Filtrado de Registros Individuales (`<clausula_donde>`, `<condiciones>`):** 
   * La cláusula `DONDE` define restricciones booleanas previas a cualquier agrupación (`WHERE`).
   * Cada `<condicion>` evalúa la relación entre un atributo (`IDENTIFICADOR`) y un literal (`NUMERO` o `CADENA`), traduciendo las frases operadoras al símbolo relacional correspondiente (`>`, `<`, `=`, `!=`).
   * La recursividad por izquierda en `<condiciones>` garantiza la evaluación secuencial de izquierda a derecha al enlazar múltiples condiciones con `Y` (`AND`) u `O` (`OR`).

3. **Agrupación y Filtrado de Grupos (`<clausula_agrupar>`, `<clausula_teniendo>`):** 
   * La instrucción `AGRUPAR POR` consolida los registros según el atributo especificado (`GROUP BY`).
   * La subcláusula opcional `TENIENDO` (`HAVING`) solo es válida dentro del contexto de una agrupación y evalúa condiciones compuestas (`<condiciones_grupo>`) sobre funciones agregadas o columnas agrupadas.

4. **Ordenamiento de Resultados (`<clausula_ordenar>`, `<direccion>`):** 
   * La cláusula `ORDENAR POR` establece el atributo de clasificación final (`ORDER BY`).
   * Si no se indica dirección ($\varepsilon$), se asume ordenamiento ascendente (`ASC`) por defecto; caso contrario, se aplica `ASC` o `DESC` según corresponda.

5. **Construcción del AST y Generación de Código:** 
   * Cada regla sintáctica ejecuta una acción semántica en `ply.yacc` que construye progresivamente un Árbol de Sintaxis Abstracta (AST) en formato diccionario/JSON. Este árbol representa la intención limpia de la consulta y sirve como entrada estructurada para que el agente LLM (Ollama) genere la sentencia SQL final sin ambigüedades.

## Implementación

In [ ]:
import json
import ply.lex as lex
import ply.yacc as yacc
import ollama

MODELO_OLLAMA = "llama3.2"

# ==========================================
# 1. ANALIZADOR LÉXICO (PLY.LEX)
# ==========================================
tokens = (
    'BUSCAR', 'DONDE', 'AGRUPAR', 'TENIENDO', 'ORDENAR', 'POR', 'DE',
    'Y', 'O', 'ASCENDENTE', 'DESCENDENTE', 'FUNCION', 'OPERADOR',
    'IDENTIFICADOR', 'NUMERO', 'CADENA',
)

t_ignore = ' \t\n'

def t_OPERADOR(t):
    r'\b(sea\ mayor\ a|mayor\ a|sea\ menor\ a|menor\ a|sea\ igual\ a|igual\ a|sea\ diferente\ de|diferente\ de)\b'
    return t

def t_FUNCION(t):
    r'\b(contar|promedio|suma|maximo|minimo)\b'
    return t

def t_ASCENDENTE(t):
    r'\b(ascendente|asc)\b'
    return t

def t_DESCENDENTE(t):
    r'\b(descendente|desc)\b'
    return t

def t_BUSCAR(t):
    r'\bbuscar\b'
    return t

def t_DONDE(t):
    r'\bdonde\b'
    return t

def t_AGRUPAR(t):
    r'\bagrupar\b'
    return t

def t_TENIENDO(t):
    r'\bteniendo\b'
    return t

def t_ORDENAR(t):
    r'\bordenar\b'
    return t

def t_POR(t):
    r'\bpor\b'
    return t

def t_DE(t):
    r'\bde\b'
    return t

def t_Y(t):
    r'\by\b'
    return t

def t_O(t):
    r'\bo\b'
    return t

def t_CADENA(t):
    r'"[^"]*"'
    t.value = t.value[1:-1]
    return t

def t_NUMERO(t):
    r'\d+(\.\d+)?'
    t.value = float(t.value) if '.' in t.value else int(t.value)
    return t

def t_IDENTIFICADOR(t):
    r'[a-zA-Z_][a-zA-Z0-9_]*'
    return t

def t_error(t):
    raise SyntaxError(f"Error léxico: Carácter no reconocido '{t.value[0]}'")

import sys
sys.modules[__name__].__file__ = "notebook.py"
lexer = lex.lex()


# ==========================================
# 2. ANALIZADOR SINTÁCTICO Y AST (PLY.YACC)
# ==========================================

#Reglas sintácticas

def p_programa(p):
    '''programa : consulta'''
    p[0] = p[1]

def p_consulta(p):
    '''consulta : BUSCAR seleccion IDENTIFICADOR clausula_donde clausula_agrupar clausula_ordenar'''
    p[0] = {
        "accion": "SELECT",
        "seleccion": p[2],
        "tabla": p[3],
        "where": p[4],
        "group_by": p[5],
        "order_by": p[6]
    }

def p_seleccion(p):
    '''seleccion : FUNCION DE
                 | empty'''
    p[0] = {"funcion": p[1]} if len(p) == 3 else "*"

def p_clausula_donde(p):
    '''clausula_donde : DONDE condiciones
                      | empty'''
    p[0] = p[2] if len(p) == 3 else None

def p_condiciones_rec(p):
    '''condiciones : condiciones conector_logico condicion'''
    p[0] = {"izquierda": p[1], "conector": p[2], "derecha": p[3]}

def p_condiciones_base(p):
    '''condiciones : condicion'''
    p[0] = p[1]

def p_condicion(p):
    '''condicion : IDENTIFICADOR OPERADOR valor'''
    p[0] = {"campo": p[1], "operador": p[2], "valor": p[3]}

def p_clausula_agrupar(p):
    '''clausula_agrupar : AGRUPAR POR IDENTIFICADOR clausula_teniendo
                        | empty'''
    p[0] = {"campo": p[3], "having": p[4]} if len(p) == 5 else None

def p_clausula_teniendo(p):
    '''clausula_teniendo : TENIENDO condiciones_grupo
                         | empty'''
    p[0] = p[2] if len(p) == 3 else None

def p_condiciones_grupo_rec(p):
    '''condiciones_grupo : condiciones_grupo conector_logico condicion_grupo'''
    p[0] = {"izquierda": p[1], "conector": p[2], "derecha": p[3]}

def p_condiciones_grupo_base(p):
    '''condiciones_grupo : condicion_grupo'''
    p[0] = p[1]

def p_condicion_grupo(p):
    '''condicion_grupo : FUNCION OPERADOR valor
                       | IDENTIFICADOR OPERADOR valor'''
    p[0] = {"objetivo": p[1], "operador": p[2], "valor": p[3]}

def p_clausula_ordenar(p):
    '''clausula_ordenar : ORDENAR POR IDENTIFICADOR direccion
                        | empty'''
    p[0] = {"campo": p[3], "direccion": p[4]} if len(p) == 5 else None

def p_direccion(p):
    '''direccion : ASCENDENTE
                 | DESCENDENTE
                 | empty'''
    p[0] = p[1] if p[1] else "ascendente"

def p_conector_logico(p):
    '''conector_logico : Y
                       | O'''
    p[0] = p[1]

def p_valor(p):
    '''valor : NUMERO
             | CADENA'''
    p[0] = p[1]

def p_empty(p):
    '''empty :'''
    p[0] = None

def p_error(p):
    if p:
        raise SyntaxError(f"Error de sintaxis cerca de '{p.value}' (token {p.type})")
    raise SyntaxError("Error de sintaxis: fin de entrada inesperado")


parser = yacc.yacc(write_tables=False, debug=False, tabmodule=None)

# ==========================================
# 3. AGENTE OLLAMA (GENERACIÓN DE SQL)
# ==========================================
def traducir_a_sql(consulta_dsl, modelo="llama3.2"):
    ast = parser.parse(consulta_dsl, lexer=lexer)
    ast_json = json.dumps(ast, indent=2, ensure_ascii=False)
    
    prompt = f"""Sos un compilador estricto de AST (JSON) a SQL estándar.
Convertí el siguiente árbol sintáctico a una consulta SQL válida.
Mapeos obligatorios:
- "contar" -> COUNT(*), "promedio" -> AVG(*), "suma" -> SUM(*), "maximo" -> MAX(*), "minimo" -> MIN(*)
- "sea mayor a"/"mayor a" -> >, "sea menor a"/"menor a" -> <, "sea igual a"/"igual a" -> =, "sea diferente de"/"diferente de" -> !=
- "y" -> AND, "o" -> OR
- "ascendente"/"asc" -> ASC, "descendente"/"desc" -> DESC
- Si un campo en el AST es null (como "where", "group_by" o "order_by"), NO incluyas esa cláusula en el SQL.

Respondé ÚNICAMENTE con la sentencia SQL terminada en punto y coma. Sin bloques markdown ni explicaciones.

AST:
{ast_json}
"""
    respuesta = ollama.chat(
        model=modelo,
        messages=[{"role": "user", "content": prompt}]
    )
    return ast_json, respuesta['message']['content'].strip()

## Tests

In [11]:
def ejecutar_testsuite():
    print("=== INICIANDO SUITE DE PRUEBAS (10 CASOS) ===\n")

    # --- PROGRAMAS VÁLIDOS (7 CASOS) ---w

    # Test 1: Consulta mínima (solo SELECT * FROM)
    t1 = parser.parse("buscar clientes", lexer=lexer)
    assert t1["tabla"] == "clientes" and t1["seleccion"] == "*"
    print("✔ Test 1 (Válido - Consulta simple): PASÓ")

    # Test 2: Selección con función de agregación (COUNT)
    t2 = parser.parse("buscar contar de ventas", lexer=lexer)
    assert t2["seleccion"] == {"funcion": "contar"} and t2["tabla"] == "ventas"
    print("✔ Test 2 (Válido - Función en SELECT): PASÓ")

    # Test 3: Filtro WHERE simple con número
    t3 = parser.parse("buscar productos donde precio sea menor a 5000", lexer=lexer)
    assert t3["where"] == {"campo": "precio", "operador": "sea menor a", "valor": 5000}
    print("✔ Test 3 (Válido - Filtro WHERE numérico): PASÓ")

    # Test 4: Filtro WHERE compuesto con cadenas y conectores lógicos (y / o)
    t4 = parser.parse('buscar usuarios donde rol igual a "admin" y activo igual a 1 o estado diferente de "baja"', lexer=lexer)
    assert t4["where"]["conector"] == "o"
    print("✔ Test 4 (Válido - Múltiples condiciones lógicas): PASÓ")

    # Test 5: Agrupación (GROUP BY) y filtro de grupo (HAVING / teniendo)
    t5 = parser.parse("buscar promedio de productos agrupar por marca teniendo promedio sea mayor a 1000", lexer=lexer)
    assert t5["group_by"]["campo"] == "marca" and t5["group_by"]["having"]["valor"] == 1000
    print("✔ Test 5 (Válido - GROUP BY + HAVING): PASÓ")

    # Test 6: Ordenamiento explícito (ORDER BY DESC)
    t6 = parser.parse("buscar empleados ordenar por salario descendente", lexer=lexer)
    assert t6["order_by"] == {"campo": "salario", "direccion": "descendente"}
    print("✔ Test 6 (Válido - ORDER BY descendente): PASÓ")

    # Test 7: Traducción End-to-End completa con Agente Ollama
    consulta_e2e = 'buscar suma de ventas donde sucursal sea igual a "Norte" agrupar por vendedor teniendo suma mayor a 10000 ordenar por vendedor ascendente'
    _, sql_e2e = traducir_a_sql(consulta_e2e)
    assert "SELECT" in sql_e2e.upper() and "GROUP BY" in sql_e2e.upper() and "HAVING" in sql_e2e.upper()
    print(f"✔ Test 7 (Válido - Traducción E2E con Ollama): PASÓ\n   -> SQL generado: {sql_e2e}")

    # --- CASOS DE ERROR (3 CASOS) ---

    # Test 8: Error Léxico (carácter ilegal no definido en el alfabeto, ej: @ o $)
    try:
        parser.parse("buscar productos donde precio sea mayor a $500", lexer=lexer)
        assert False, "Debió fallar por error léxico"
    except SyntaxError as e:
        print(f"✔ Test 8 (Error Léxico detectado correctamente): PASÓ -> {e}")

    # Test 9: Error Sintáctico (falta el identificador de la tabla después de 'buscar')
    try:
        parser.parse("buscar donde precio mayor a 100", lexer=lexer)
        assert False, "Debió fallar por error sintáctico"
    except SyntaxError as e:
        print(f"✔ Test 9 (Error Sintáctico detectado correctamente): PASÓ -> {e}")

    # Test 10: Error Sintáctico / Estructural (cláusula 'teniendo' sin 'agrupar por' previo)
    try:
        parser.parse("buscar productos teniendo contar mayor a 5", lexer=lexer)
        assert False, "Debió fallar porque 'teniendo' requiere 'agrupar por'"
    except SyntaxError as e:
        print(f"✔ Test 10 (Error Sintáctico de orden de cláusulas): PASÓ -> {e}")

    print("\n=== TODOS LOS TESTS (10/10) SE EJECUTARON CON ÉXITO ===")

# Ejecutamos la suite para que quede la salida impresa en la notebook
ejecutar_testsuite()

=== INICIANDO SUITE DE PRUEBAS (10 CASOS) ===

✔ Test 1 (Válido - Consulta simple): PASÓ
✔ Test 2 (Válido - Función en SELECT): PASÓ
✔ Test 3 (Válido - Filtro WHERE numérico): PASÓ
✔ Test 4 (Válido - Múltiples condiciones lógicas): PASÓ
✔ Test 5 (Válido - GROUP BY + HAVING): PASÓ
✔ Test 6 (Válido - ORDER BY descendente): PASÓ
✔ Test 7 (Válido - Traducción E2E con Ollama): PASÓ
   -> SQL generado: SELECT SUM(ventas.sucursal) AS suma FROM ventas WHERE ventas.sucursal = 'Norte' GROUP BY ventas.vendedor HAVING SUM(ventas.vendedor) > 10000 ORDER BY ventas.vendedor ASC;
✔ Test 8 (Error Léxico detectado correctamente): PASÓ -> Error léxico: Carácter no reconocido '$'
✔ Test 9 (Error Sintáctico detectado correctamente): PASÓ -> Error de sintaxis cerca de 'donde' (token DONDE)
✔ Test 10 (Error Sintáctico de orden de cláusulas): PASÓ -> Error de sintaxis cerca de 'teniendo' (token TENIENDO)

=== TODOS LOS TESTS (10/10) SE EJECUTARON CON ÉXITO ===


### **Otros tests**:

In [9]:
ast, sql = traducir_a_sql("buscar clientes")
print(sql)

ast, sql = traducir_a_sql("buscar ventas donde precio sea mayor a 50")
print(sql)

SELECT * FROM clientes;
SELECT * FROM ventas WHERE precio > 50;


In [ ]:
ast, sql = traducir_a_sql('buscar clientes donde edad sea mayor a 24')
print(sql)

ast, sql = traducir_a_sql('buscar clientes donde nombre sea igual a "adriel"')
print(sql)

ast, sql = traducir_a_sql('buscar persona donde nombre sea igual a "adriel" y edad sea igual a 24')
print(sql)

ast, sql = traducir_a_sql('buscar edificio donde altura sea mayor a 50 y material sea igual a "concreto"')
print(sql)

ast, sql = traducir_a_sql('buscar pago donde fecha sea igual a "18/03/2002" y monto sea mayor a 5000')
print(sql)

ast, sql = traducir_a_sql('buscar clientes donde deudas sea mayor a 1 agrupar por ciudad')
print(sql)

ast, sql = traducir_a_sql('buscar ventas donde producto sea igual a "laptop" y cantidad sea mayor a 10 agrupar por vendedor teniendo suma sea mayor a 10000 ordenar por vendedor ascendente')
print(sql)

ast, sql = traducir_a_sql('buscar suma de ventas donde sucursal sea igual a "Norte" agrupar por vendedor teniendo suma sea mayor a 10000 ordenar por vendedor ascendente')
print(sql)

ast, sql = traducir_a_sql('buscar promedio de ventas agrupar por vendedor teniendo promedio sea mayor a 10000 ordenar por vendedor')
print(sql)

SELECT * FROM clientes WHERE edad > 24
SELECT * FROM clientes WHERE nombre = 'adriel';
SELECT * FROM persona WHERE nombre = 'adriel' AND edad = 24
SELECT * FROM edificio WHERE altura > 50 AND material = 'concreto'
SELECT * FROM pago WHERE fecha = '18/03/2002' AND monto > 5000
SELECT * FROM clientes WHERE deudas > 1 GROUP BY ciudad
SELECT * FROM ventas WHERE producto = 'laptop' AND cantidad > 10 GROUP BY vendedor HAVING suma(cantidad) > 10000 ORDER BY vendedor ASC
SELECT SUM(vendedor) FROM ventas WHERE sucursal = 'Norte' GROUP BY vendedor HAVING SUM(vendedor) > 10000 ORDER BY vendedor ASC;
